# Projektumgebung einrichten

**Aufgabe:** Die Projektpakete installieren und den ausgewählten Python-Kernel prüfen.

1. Oben rechts **Select Kernel → Python Environments → .venv** wählen. Eine vorhandene passende Conda-Umgebung funktioniert ebenfalls.
2. Die Umgebungszelle und die Installationszelle ausführen.
3. Danach **Restart Kernel** wählen und nur die abschließende Prüfzelle ausführen.

Voraussetzung ist Python **3.10 oder neuer**; Python 3.14 ist bereits im Projekt getestet.
Es ist keine Conda-Umgebung mit dem Namen `pybis` erforderlich.
Dieses Notebook liest keine Credentials und verbindet sich nicht mit openBIS.

Falls VS Code noch keinen Kernel starten kann, **Install ipykernel** im VS-Code-Hinweis wählen.
Alternativ einmalig im Repository-Terminal: `.\.venv\Scripts\python -m pip install ipykernel`.
Fehlt `.venv` vollständig, vorher `python -m venv .venv` im Terminal ausführen.

In [ ]:
import sys
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)
print("Umgebung:", Path(sys.prefix))
if sys.version_info < (3, 10):
    raise RuntimeError("GaSi benötigt Python 3.10 oder neuer. In VS Code einen passenden Kernel auswählen.")
print("Python ist geeignet. Eine Conda-Umgebung oder ein bestimmter Umgebungsname ist nicht erforderlich.")

## Requirements im aktiven Kernel installieren

Die folgende Zelle installiert `requirements-dev.txt` einschließlich pyBIS, Jupyter-Kernel,
HTML-Paketen und Prüfwerkzeugen. Dafür wird Zugriff auf den konfigurierten Paketindex benötigt.
Der Befehl wird als Argumentliste ohne Shell ausgeführt.

In [ ]:
from pathlib import Path
import subprocess
import sys

if sys.version_info < (3, 10):
    raise RuntimeError("Für die Projektpakete zuerst einen Kernel mit Python 3.10 oder neuer auswählen.")

# Funktioniert auch, wenn das Notebook aus einem Unterordner geöffnet wurde.
working_dir = Path.cwd().resolve()
repo_root = next((p for p in (working_dir, *working_dir.parents)
                  if (p / "requirements-dev.txt").is_file()
                  and (p / "requirements-playground.txt").is_file()
                  and (p / "requirements-docs.txt").is_file()), None)
if repo_root is None:
    raise RuntimeError("Repository nicht gefunden. Das Notebook im Ordner gasi-openbis öffnen.")

requirements_file = repo_root / "requirements-dev.txt"
print("Installiere Projektpakete für:", sys.executable)
print("Requirements:", requirements_file)
# Entspricht %pip install -r requirements-dev.txt; verwendet sicher den aktiven Kernel.
installation = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements_file)],
    check=True,
)
print("Installation abgeschlossen. Kernel neu starten und anschließend die Prüfzelle unten ausführen.")

## Nach dem Kernel-Neustart prüfen

Diese Zelle funktioniert auch allein mit einem frischen Kernel. Sie prüft die installierten
Versionen **und tatsächliche Imports**. Anschließend ist die Projektumgebung eingerichtet.

Für das Workshop-Beispiel nur dessen Einrichtungszellen ausführen, wenn du ausschließlich
die Umgebung prüfen möchtest. Spätere Workshop-Zellen enthalten Schreibübungen.

In [ ]:
import importlib
from importlib.metadata import PackageNotFoundError, version
import sys

print("Geprüfter Interpreter:", sys.executable)
missing = []
packages = {"pybis": "pybis", "ipykernel": "ipykernel", "Markdown": "markdown",
            "ipywidgets": "ipywidgets", "nbclient": "nbclient", "nbformat": "nbformat"}
for distribution, module in packages.items():
    try:
        installed_version = version(distribution)
        importlib.import_module(module)
    except (PackageNotFoundError, ImportError):
        missing.append(distribution)
        print(distribution + ": fehlt oder lässt sich nicht importieren")
    else:
        print(f"{distribution}: {installed_version} – Import erfolgreich")
if missing:
    raise RuntimeError("Projektpakete fehlen im aktiven Kernel: " + ", ".join(missing)
                       + ". Installationszelle ausführen, Kernel neu starten und diese Zelle wiederholen.")
print("Projektumgebung bereit. Weiter mit GaSi_00_Session_einrichten.ipynb oder den Offline-Notebooks.")